##DSAI 413 A2

## Stage 1 — Bootstrap (clone, install, secrets, symlink dataset)

In [ ]:
import os, sys, pathlib, subprocess

if not pathlib.Path('/kaggle/working/cxr-intel').exists():
    subprocess.run(['git', 'clone', 'https://github.com/jo3591/assigment2dsai413',
                    '/kaggle/working/cxr-intel'], check=True)
%cd /kaggle/working/cxr-intel
!git pull
!pip install -q -r requirements-colab.txt
!pip install -q -e .

dst = pathlib.Path('/kaggle/working/cxr-intel/data/raw')
dst.parent.mkdir(parents=True, exist_ok=True)
if not dst.exists():
    dst.symlink_to('/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset')

sys.path.insert(0, '/kaggle/working/cxr-intel/src')
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for k in ['HF_TOKEN', 'NVIDIA_API_KEY', 'KAGGLE_USERNAME', 'KAGGLE_KEY']:
    try: os.environ[k] = s.get_secret(k)
    except Exception: pass
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['QA_SYNTH_MODEL'] = 'meta/llama-3.1-8b-instruct'
os.environ['QA_JUDGE_MODEL'] = 'meta/llama-3.3-70b-instruct'

import torch
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

## Stage 2 — Data preprocessing (~3 min)

Extracts reports, computes CheXpert labels, builds patient-disjoint train/val/test splits.

In [ ]:
!python -u scripts/download_data.py --config configs/data.yaml --skip-download --no-basename-index

## Stage 3 — Synthesize QA dataset (~90 min, NVIDIA NIM)

~6,200 train + ~800 test validated QA pairs.

In [ ]:
!python -u scripts/build_qa_dataset.py --config configs/data.yaml

## Stage 4 — Build BiomedCLIP index (~5 min)

In [ ]:
!python -u scripts/build_indices.py --backend biomedclip

## Stage 5 — Patch ColPali v1.3 adapter + build ColPali index (~30 min)

Transformers ≥4.51 renamed PaliGemma module paths (`model.language_model.model.X` → `model.model.language_model.X`). The official `vidore/colpali-v1.3` LoRA adapter is keyed against the old paths, so we patch it offline so the LoRA weights merge correctly.

In [ ]:
import shutil, safetensors.torch
from pathlib import Path
from huggingface_hub import snapshot_download

src = Path(snapshot_download('vidore/colpali-v1.3'))
out = Path('/kaggle/working/colpali-v1.3-patched')
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True)
for f in src.iterdir():
    if f.is_file():
        shutil.copy2(f, out / f.name)

state = safetensors.torch.load_file(str(out / 'adapter_model.safetensors'))
remapped = {k.replace('model.language_model.model.', 'model.model.language_model.'): v for k, v in state.items()}
safetensors.torch.save_file(remapped, str(out / 'adapter_model.safetensors'))
print(f'\u2713 Patched {len(remapped)} adapter keys \u2192 {out}')

In [ ]:
# bitsandbytes 0.44 has a Triton incompatibility that breaks ColPali loading via peft's lora.bnb dispatch
!pip uninstall -y bitsandbytes

import pandas as pd, torch
from cxr_intel.retrieval.colpali_index import ColPaliRetriever

df = pd.read_parquet('data/processed/reports.parquet')
items = [
    {'study_id': str(r['study_id']),
     'image_path': str(r['image_path']),
     'report_text': (str(r.get('findings','')) + ' ' + str(r.get('impression',''))).strip()}
    for _, r in df.iterrows()
]
r = ColPaliRetriever(
    checkpoint='/kaggle/working/colpali-v1.3-patched',
    torch_dtype='float16',
    device_map='auto',
    batch_size=2,
)
r.index(items, 'data/indices/colpali_zs')
print(f'\u2713 ColPali index saved | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Stage 6 — Restart kernel, then prep eval

**Restart the kernel now** (Run → Restart & clear cell outputs) to free GPU memory held by the ColPali model used for indexing. After the restart, re-run **only Stage 1** (Bootstrap), then continue from this cell.

In [ ]:
# Force newer transformers (for MedGemma's gemma3) + reinstall bitsandbytes for INT4 MedGemma
!pip install -q --force-reinstall --no-deps 'transformers>=4.50,<4.53'
!pip install -q --upgrade 'bitsandbytes>=0.45,<0.50'

# Patch run_eval.py: (a) INT4 MedGemma quantization, (b) point ColPali at the patched checkpoint,
# (c) use NVIDIA NIM model-name format for the LLM judge (slash, not dash)
from pathlib import Path
p = Path('scripts/run_eval.py')
content = p.read_text()
content = content.replace(
    'medgemma = MedGemmaRunner()\n    medgemma.load()',
    "medgemma = MedGemmaRunner(quantization='int4')\n    medgemma.load()"
).replace(
    'r = ColPaliRetriever(name="colpali_zs")',
    'r = ColPaliRetriever(name="colpali_zs", checkpoint="/kaggle/working/colpali-v1.3-patched")',
).replace(
    'judge = LLMJudge(LLMRouter())',
    "judge = LLMJudge(LLMRouter(model='meta/llama-3.1-8b-instruct'))"
)
p.write_text(content)

# Patch metrics_report.py to use sacrebleu's BLEU class API (new in 2.4+)
mr = Path('src/cxr_intel/eval/metrics_report.py')
mrc = mr.read_text()
if 'from sacrebleu.metrics import BLEU' not in mrc:
    mrc = mrc.replace(
        'import sacrebleu\n\n    bleu1 = sacrebleu.corpus_bleu(preds, [refs], max_ngram_order=1).score / 100',
        'from sacrebleu.metrics import BLEU\n\n    bleu1 = BLEU(max_ngram_order=1, effective_order=True).corpus_score(preds, [refs]).score / 100'
    ).replace(
        'bleu2 = sacrebleu.corpus_bleu(preds, [refs], max_ngram_order=2).score / 100',
        'bleu2 = BLEU(max_ngram_order=2, effective_order=True).corpus_score(preds, [refs]).score / 100'
    ).replace(
        'bleu4 = sacrebleu.corpus_bleu(preds, [refs]).score / 100',
        'bleu4 = BLEU(effective_order=True).corpus_score(preds, [refs]).score / 100'
    )
    mr.write_text(mrc)
print('\u2713 Patches applied')

## Stage 7 — Full evaluation (~90 min)

3 configs × 3 modes on 50 patient-disjoint test cases.

In [ ]:
!python -u scripts/run_eval.py \
    --mode report --mode qa --mode retrieval \
    --configs medgemma_only,biomedclip_rag,colpali_zs_rag \
    --test-size 50 --top-k 3

## Stage 8 — Inspect results

These are the actual numbers from the eval run on May 16 2026.

In [ ]:
import pandas as pd
for name in ['report_metrics', 'qa_metrics', 'retrieval_metrics']:
    print(f'\n=== {name} ===')
    print(pd.read_csv(f'results/tables/{name}.csv').to_string(index=False))


=== report_metrics ===
        config    bleu1    bleu2    bleu4  rouge_l  bertscore_f1  chexbert_f1  radgraph_f1  n
 medgemma_only 0.000740 0.000490 0.000214 0.185579      0.843089     0.300778          NaN 50
biomedclip_rag 0.002391 0.001829 0.001152 0.261160      0.863751     0.352333          NaN 50
colpali_zs_rag 0.003564 0.002711 0.001642 0.259949      0.865022     0.428788          NaN 50

=== qa_metrics ===
        config  exact_match  token_f1  bertscore_f1  llm_judge_mean  llm_judge_pass_rate  n
 medgemma_only          0.0  0.259093      0.887048            3.42                 0.66 50
biomedclip_rag          0.0  0.448314      0.909041            4.39                 0.88 50
colpali_zs_rag          0.0  0.453927      0.911461            4.34                 0.90 50

=== retrieval_metrics ===
   backend  recall_at_1  recall_at_5  recall_at_10    mrr  ndcg_at_10  n                       by_k
biomedclip          0.0          0.0          0.02 0.0025    0.006309 50 {1: 0.0, 5: 

In [ ]:
# All-in-one Gradio demo cell — paste in a fresh Kaggle notebook with same data attached
!pip install -q gradio pyngrok

import os, sys, pathlib
sys.path.insert(0, '/kaggle/working/cxr-intel/src')
os.chdir('/kaggle/working/cxr-intel')

from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
os.environ['NVIDIA_API_KEY'] = s.get_secret('NVIDIA_API_KEY')
os.environ['LLM_PROVIDER'] = 'nvidia'
ngrok_token = s.get_secret('NGROK_TOKEN')

from cxr_intel.models.medgemma_runner import MedGemmaRunner
from cxr_intel.retrieval.colpali_index import ColPaliRetriever
from cxr_intel.generation.report_pipeline import ReportPipeline
from cxr_intel.generation.qa_pipeline import QAPipeline

medgemma = MedGemmaRunner(quantization='int4'); medgemma.load()
colpali = ColPaliRetriever(checkpoint='/kaggle/working/colpali-v1.3-patched', torch_dtype='float16')
colpali.load('data/indices/colpali_zs')

import gradio as gr
def report(img):
    out = ReportPipeline('colpali_zs_rag', retriever=colpali, medgemma=medgemma, top_k=3).run(img)
    return out.report_text
def qa(img, q):
    out = QAPipeline('colpali_zs_rag', retriever=colpali, medgemma=medgemma, top_k=3).run(img, q)
    return out.answer

with gr.Blocks() as demo:
    gr.Markdown('# CXR Intelligence — DSAI 413 A2')
    with gr.Tab('Report'):
        i = gr.Image(type='pil'); o = gr.Textbox(lines=8); gr.Button('Generate').click(report,[i],[o])
    with gr.Tab('QA'):
        i2 = gr.Image(type='pil'); q2 = gr.Textbox(label='Question'); o2 = gr.Textbox(lines=4)
        gr.Button('Answer').click(qa,[i2,q2],[o2])

from pyngrok import ngrok
ngrok.set_auth_token(ngrok_token)
url = ngrok.connect(7860).public_url
print(f'\n🌐 DEMO URL: {url}\n')
demo.launch(server_name='0.0.0.0', server_port=7860, quiet=True)
